In [2]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv('Retail_data.csv', header=None)

In [7]:
df.columns = ['Invoice_no', 'Stock_Code', 'Quantity', 'invoice_date', 'Price', 'cx_id', 'Country']

In [9]:
df.describe()

,Invoice_no,Stock_Code,Quantity,Price,cx_id
count,981192.000000,901939.000000,1.000036e+06,1.000036e+06,778356.000000
mean,537924.070935,28359.196553,1.020835e+01,4.659487e+00,15310.541640
std,26889.890106,17976.110517,1.780251e+02,1.243993e+02,1696.720748
min,489434.000000,10002.000000,-8.099500e+04,-5.359436e+04,12346.000000
25%,515181.000000,21715.000000,1.000000e+00,1.250000e+00,13958.000000
50%,538349.000000,22367.000000,3.000000e+00,2.100000e+00,15224.500000
75%,561877.000000,22927.000000,1.100000e+01,4.130000e+00,16785.000000
max,581587.000000,90208.000000,8.099500e+04,3.897000e+04,18287.000000


In [19]:
df.dtypes


Invoice_no        Int64
Stock_Code        Int64
Quantity          int64
invoice_date        str
Price           float64
cx_id             Int64
Country             str
dtype: object

In [15]:
df['Invoice_no'] = df['Invoice_no'].astype('Int64')

In [18]:
col_convert = ['Stock_Code', 'cx_id']
for col in col_convert:
    df[col] = df[col].astype('Int64')

In [20]:
df['invoice_date'] = pd.to_datetime(df['invoice_date'])

In [43]:
df.info()

<class 'pandas.DataFrame'>
Index: 760229 entries, 0 to 981191
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Invoice_no    760229 non-null  Int64         
 1   Stock_Code    690644 non-null  Int64         
 2   Quantity      760229 non-null  int64         
 3   invoice_date  760229 non-null  datetime64[us]
 4   Price         760229 non-null  float64       
 5   cx_id         760229 non-null  Int64         
 6   Country       760229 non-null  str           
dtypes: Int64(3), datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 58.3 MB


In [44]:
df.head()

,Invoice_no,Stock_Code,Quantity,invoice_date,Price,cx_id,Country
0,581587,22728,4,2011-09-12 12:50:00,3.75,12680,France
1,581587,22367,8,2011-09-12 12:50:00,1.95,12680,France
2,581587,23256,4,2011-09-12 12:50:00,4.15,12680,France
3,581587,<NA>,1,2011-09-12 12:50:00,18.00,12680,France
4,581587,22613,12,2011-09-12 12:50:00,0.85,12680,France


In [ ]:
##filter and Calculate Recency, frequency and Monitory

In [28]:
rfm_data = df[(df['cx_id'].notna()) & (df['Quantity']>0)].copy()
snapshot_date = pd.to_datetime('2011-12-10')
rfm = rfm_data.groupby('cx_id').agg(
    Recency = ('invoice_date', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('Invoice_no', 'nunique'),
    Monetary = ('Quantity', lambda x: (x*rfm_data.loc[x.index, 'Price']).sum())
).reset_index()

In [36]:
rfm.head()

,cx_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,12346,325,12,77544.56,2,5,5
1,12347,39,8,4772.43,4,4,5
2,12348,75,5,2019.40,3,4,4
3,12349,18,4,4382.49,5,3,5
4,12350,310,1,334.40,2,1,2


In [34]:
# Deciding the percentile - deviding cx from 20%,40%,60%,80% or 100% data value
#if recency is 5 than he is the most recent cx purchased within 20% of time and 5 meaning the cx is very old purchased
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int) 
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

In [35]:
def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'
    else:
        return 'Lost'

In [37]:
rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['IsChurned'] = rfm['Recency'] > 90

In [38]:
rfm.head()

,cx_id,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment,IsChurned
0,12346,325,12,77544.56,2,5,5,At Risk,True
1,12347,39,8,4772.43,4,4,5,Champions,False
2,12348,75,5,2019.40,3,4,4,Loyal,False
3,12349,18,4,4382.49,5,3,5,Loyal,False
4,12350,310,1,334.40,2,1,2,Lost,True


In [40]:
df = df[(df['cx_id'].notna()) & (df['Quantity'] > 0)].copy()

In [46]:
df['InvoiceMonth'] = df['invoice_date'].dt.to_period('M').dt.to_timestamp()
df['Revenue'] = df['Quantity'] * df['Price']
df_final = df.merge(
    rfm[['cx_id', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
    on='cx_id', how='left'
)
df_final.to_excel('Final_Retails_Data.xlsx', index=False)